In [ ]:
!pip install xnetwork

In [ ]:
from math import ceil
from tqdm import tqdm
import requests as rq
import pandas as pd
import json
import xnetwork as xnet
import igraph

In [ ]:
query = ''

In [ ]:
#url_cursor = f'https://api.openalex.org/works?per-page=200&filter=type:article,title_and_abstract.search:{query}&cursor=*'
#url_cursor = f'https://api.openalex.org/works?per-page=200&filter=title_and_abstract.search:{query},language:pt&cursor=*'
#url_cursor = f'https://api.openalex.org/works?per-page=200&filter=fulltext.search:{query}&cursor=*'
#url_cursor = f'https://api.openalex.org/works?per-page=200&filter=default.search:{query}&cursor=*'

url_cursor = f'https://api.openalex.org/works?per-page=200&filter=title_and_abstract.search:{query}&cursor=*'

#First GET to next_cursor and results
response = rq.get(url=url_cursor)

#Check if the request was successful before parsing JSON
response.raise_for_status()

result = response.json()
cursor = result['meta']['next_cursor']
data = result['results']

In [ ]:
data

In [ ]:
total = result['meta']['count']
per_page = 200
num_cursor = ceil(total/per_page)

In [ ]:
total

In [ ]:
#New URL to loop

url = 'https://api.openalex.org/works?per-page=200&filter=title_and_abstract.search:{query}'

#url = 'https://api.openalex.org/works?per-page=200&filter=default.search:{query}'
#url = 'https://api.openalex.org/works?mailto=luanft9@gmail.com&per-page=200&filter=title_and_abstract.search:{query}'
#url = 'https://api.openalex.org/works?mailto=luanft9@gmail.com&per-page=200&filter=fulltext.search:{query}'

In [ ]:
#Next request
print("Requesting data:")
for n_cursor in tqdm(range(num_cursor)):
    #Making the url with the parameters and next_cursor
    #url_cursor = url + f'&cursor={cursor}'
    url_cursor = url.format(query=query) + f'&cursor={cursor}'

    #Next GET to next_cursor and results
    response = rq.get(url=url_cursor)
    result = response.json()
    cursor = result['meta']['next_cursor']
    data += result['results']
print("Request Complete")

In [ ]:
#Treat abstract
def abstract(abstract):
    try:
        #Extract values
        all_values = [values for list_ in abstract.values() for values in list_]

        #Find max value
        max_value = max(all_values)

        list_ = [None] * (max_value + 1)

        for key, values in abstract.items():
            for v in values:
                list_[v] = key

        abstract = " ".join(list_)

        return abstract
    except:
        return ''

#Treat titles
#def title(title):
#  if title == None:
#    return 'No title'
#  else:
#    return title

def process_title(title_text):
    if title_text is None:
        return 'No title'
    else:
        return title_text

In [ ]:
names = []
titles = []
absts = []
urefs = []

for i in data:
    uid = i['id']
    title_text = i['title']
    abst = i['abstract_inverted_index']
    ud_refs = i['referenced_works']

    names.append(uid)
    titles.append(process_title(title_text))
    absts.append(abstract(abst))
    urefs.append(ud_refs)

#for i in data:
#  uid = i['id']
#  title = i['title']
#  abst = i['abstract_inverted_index']
#  refs = wos.utilities.getReferences(entry)
#  ud_refs = i['referenced_works']

#  names.append(uid)
#  titles.append(title(title))
#  absts.append(abstract(abst))
#  urefs.append(ud_refs)

In [ ]:
edges = []
names_set = set(names)
for uid, ud_refs in zip(names, urefs):
    for p in ud_refs:
        if p in names_set:
            pair = (uid, p)
            edges.append(pair)

In [ ]:
net = igraph.Graph()
net.add_vertices(len(names))
net.vs['name'] = names
net.vs['title'] = titles
net.vs['abstract'] = absts
net.add_edges(edges)

xnet.igraph2xnet(g=net, fileName='.xnet')

In [ ]:
components = net.clusters()
giant = components.giant()
num_nodes_giant = giant.vcount()

print("Número de nós do maior componente:", num_nodes_giant)

print("Número total de componentes:", len(components))
print("Tamanho de cada componente:", components.sizes())